# Standalone Database Table Exporter

This notebook connects to a PostgreSQL database, lists available tables in the `public` schema, allows you to select one, and exports the complete table data to a CSV file.

**This version is self-contained and does not require the `iris` project structure.** You only need `pandas`, `psycopg2`, and `ipywidgets` installed.

## 1. Configuration

**Important:** Update the `DB_PARAMS` dictionary below with your actual database connection details.

In [ ]:
# --- Database Connection Parameters --- 
# !!! MODIFY THESE VALUES TO MATCH YOUR DATABASE SETUP !!!
DB_PARAMS = {
    "host": "localhost",      # e.g., 'localhost' or an IP address
    "port": "5432",           # Default PostgreSQL port
    "dbname": "maven-finance",  # Your database name
    "user": "iris_dev",       # Your database username
    "password": ""             # Your database password (leave empty if none)
}

# --- Export Directory ---
EXPORT_DIRECTORY = "." # Export CSV to the current directory where the notebook server is running
# --- End Configuration ---

## 2. Setup and Imports

In [ ]:
import pandas as pd
import psycopg2
import psycopg2.extras # Required for UUID handling if needed
from psycopg2 import sql # Import the sql module
import ipywidgets as widgets
from IPython.display import display, FileLink
import os
import logging
import uuid

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

# Register UUID adapter (optional, but good practice if you have UUID columns)
try:
    psycopg2.extras.register_uuid()
except Exception as e:
    logger.warning(f"Could not register UUID adapter: {e}")

## 3. Database Connection Function

In [ ]:
def connect_to_db_standalone(db_params):
    """
    Connects to the PostgreSQL database using the provided parameters.

    Args:
        db_params: Dictionary with database connection parameters 
                   (host, port, dbname, user, password).

    Returns:
        Database connection object or None if connection fails.
    """
    try:
        logger.info(f"Connecting to database with parameters: host={db_params.get('host')}, " +
                   f"port={db_params.get('port')}, dbname={db_params.get('dbname')}, user={db_params.get('user')}")
        conn = psycopg2.connect(**db_params)
        conn.autocommit = False # Good practice to manage transactions explicitly if needed
        logger.info("Database connection successful")
        return conn
    except Exception as e:
        logger.error(f"Error connecting to database: {e}", exc_info=True)
        return None

## 4. List Tables and Create Widgets

In [ ]:
conn = None
table_list = []

try:
    logger.info(f"Attempting to connect to database...")
    conn = connect_to_db_standalone(DB_PARAMS)

    if conn:
        logger.info("Database connection successful.")
        with conn.cursor() as cur:
            # Query to get all table names from the public schema
            cur.execute("""
                SELECT table_name
                FROM information_schema.tables
                WHERE table_schema = 'public'
                ORDER BY table_name;
            """)
            table_list = [row[0] for row in cur.fetchall()]
            logger.info(f"Found {len(table_list)} tables in public schema.")
    else:
        logger.error("Failed to establish database connection. Check DB_PARAMS.")

except Exception as e:
    logger.error(f"An error occurred during database connection or table listing: {e}", exc_info=True)
finally:
    # Keep connection open for export, close later in the export function
    # Close the initial connection if it's still open and wasn't used for listing
    if conn and not conn.closed and not table_list:
         try:
             conn.close()
             logger.info("Initial connection closed as table listing failed.")
         except Exception as close_e:
             logger.error(f"Error closing initial connection: {close_e}")

# Widgets for table selection and export
if not table_list:
    print("Could not retrieve table list. Please check connection parameters and logs.")
    table_selector = widgets.Label("No tables found or connection failed.")
    export_button = widgets.Button(description="Export Table", disabled=True)
    output_area = widgets.Output()
else:
    table_selector = widgets.Dropdown(
        options=table_list,
        description='Select Table:',
        disabled=False,
        style={'description_width': 'initial'}
    )

    export_button = widgets.Button(
        description="Export Table",
        disabled=False,
        button_style='info', # 'success', 'info', 'warning', 'danger' or ''
        tooltip='Click to export the selected table to CSV',
        icon='download'
    )

    output_area = widgets.Output()

def on_export_button_clicked(b):
    global conn # Need access to the connection
    selected_table = table_selector.value
    output_area.clear_output()
    export_conn = None # Use a separate connection for export to ensure it's fresh

    if not selected_table:
        with output_area:
            print("Please select a table.")
        return

    # Establish a new connection for the export operation
    with output_area:
        print("Connecting to database for export...")
    export_conn = connect_to_db_standalone(DB_PARAMS)

    if not export_conn:
         with output_area:
             print("Connection failed for export. Check DB_PARAMS and database status.")
         return
    logger.info("Connection successful for export.")

    export_filename = f"{selected_table}.csv"
    export_filepath = os.path.join(EXPORT_DIRECTORY, export_filename)

    with output_area:
        print(f"Exporting table 'public.{selected_table}' to {export_filepath}...")
        try:
            # Construct the SQL query safely by quoting the identifier first
            quoted_table_name = sql.Identifier(selected_table).as_string(export_conn)
            # Now use the quoted string in an f-string for the final query
            sql_query = f"SELECT * FROM public.{quoted_table_name}"
            
            logger.info(f"Executing query: {sql_query}")

            # Use pandas to read the SQL query into a DataFrame
            df = pd.read_sql(sql_query, export_conn)
            logger.info(f"Fetched {len(df)} rows from table '{selected_table}'.")

            # Export DataFrame to CSV
            df.to_csv(export_filepath, index=False, encoding='utf-8')
            logger.info(f"Successfully exported table '{selected_table}' to {export_filepath}")

            print(f"Export complete! {len(df)} rows saved.")
            # Provide a download link
            display(FileLink(export_filepath))

        except (Exception, psycopg2.Error) as e:
            logger.error(f"Error exporting table '{selected_table}': {e}", exc_info=True)
            print(f"Error exporting table: {e}")
            # Attempt to rollback if there was a transaction error
            try:
                if export_conn and not export_conn.closed:
                    export_conn.rollback()
                    logger.info("Transaction rolled back due to error.")
            except Exception as rollback_e:
                 logger.error(f"Error during rollback: {rollback_e}")
        finally:
            # Ensure the export connection is closed
            if export_conn and not export_conn.closed:
                export_conn.close()
                logger.info("Database connection closed after export attempt.")

# Link button click event to the function
export_button.on_click(on_export_button_clicked)

# Display the widgets
display(widgets.VBox([table_selector, export_button, output_area]))

## 5. Instructions

1.  **Configure:** Update the `DB_PARAMS` in the first code cell with your database details.
2.  **Run All Cells:** Execute all cells in the notebook (e.g., `Cell > Run All` in Jupyter).
3.  **Select Table:** Choose the table you want to export from the dropdown menu.
4.  **Export:** Click the "Export Table" button.
5.  **Download:** The table data will be saved as a CSV file in the same directory as this notebook. A download link will appear in the output area.

## 6. (Optional) Close Initial Connection Manually

The export function now uses its own connection. The initial connection used for listing tables might still be open if listing succeeded. Run the cell below to close it explicitly if needed.

In [ ]:
try:
    # Check the original 'conn' variable used for listing tables
    if 'conn' in locals() and conn and not conn.closed:
        conn.close()
        logger.info("Initial database connection explicitly closed.")
        print("Initial database connection closed.")
    else:
        print("Initial database connection is already closed or was not established.")
except NameError:
    print("Initial connection variable 'conn' not found.")
except Exception as e:
    logger.error(f"Error closing initial connection: {e}")
    print(f"Error closing initial connection: {e}")